# V2 Horse Racing — PPO + GRU Training

**Environment:** Colab Pro+ (A100/V100, high-RAM runtime)

Trains a jockey policy through 5 curriculum stages:
1. Solo straight — learn effort control
2. Solo curves — learn cornering + lane positioning
3. Multi-horse — collision avoidance, drafting
4. Self-play — tactics against frozen opponents
5. Diverse full-field — jockey style conditioning

Each stage gates on a metric before advancing.

## 1. Setup

In [ ]:
# Mount Google Drive for persistent checkpoints
from google.colab import drive
drive.mount('/content/drive')

# Project paths
DRIVE_ROOT = '/content/drive/MyDrive/hr-simulation'
CHECKPOINT_DIR = f'{DRIVE_ROOT}/checkpoints/v2'
ONNX_DIR = f'{DRIVE_ROOT}/models/v2'

import os
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
os.makedirs(ONNX_DIR, exist_ok=True)
print(f'Checkpoints: {CHECKPOINT_DIR}')
print(f'ONNX models: {ONNX_DIR}')

In [ ]:
%%bash
# Install dependencies
pip install -q \
    'ray[rllib]==2.40.0' \
    'gymnasium>=1.0.0' \
    'torch>=2.0.0' \
    'numpy>=1.26.0' \
    'onnx>=1.15.0' \
    'onnxruntime>=1.17.0' \
    'tensorboard>=2.15.0'

In [ ]:
import os, sys

# Clone from GitHub (skip if already cloned)
if not os.path.exists('/content/hr-simulation/.git'):
    !git clone https://github.com/ue-too/hr-simulation.git /content/hr-simulation
else:
    !cd /content/hr-simulation && git pull

# Install package
%cd /content/hr-simulation
!pip install -q -e .
%cd /content

# Add to path as fallback
if '/content/hr-simulation' not in sys.path:
    sys.path.insert(0, '/content/hr-simulation')

import horse_racing
print('horse_racing package loaded')

In [ ]:
# Verify GPU is available
import torch
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

import psutil
print(f'CPU cores: {psutil.cpu_count()}')
print(f'RAM: {psutil.virtual_memory().total / 1e9:.1f} GB')

In [ ]:
# Quick env smoke test
from horse_racing.env.racing_env import HorseRacingEnv
import numpy as np

env = HorseRacingEnv(
    track_path='/content/hr-simulation/tracks/curriculum_1_straight.json',
    num_horses=1,
    max_steps=100,
)
obs, info = env.reset(seed=42)
print(f'Obs shape: {obs.shape}, dtype: {obs.dtype}')
print(f'Action space: {env.action_space}')

# Run a few steps with max effort
total_reward = 0
for i in range(50):
    obs, reward, terminated, truncated, info = env.step(np.array([1.0, 0.0]))
    total_reward += reward
    if terminated:
        break

print(f'After {i+1} steps: progress={info["progress"]:.3f}, stamina={info["stamina_ratio"]:.3f}, reward={total_reward:.1f}')
print('Env smoke test PASSED')

## 2. Training Infrastructure

In [ ]:
import ray
from ray import tune
from ray.rllib.algorithms.ppo import PPOConfig
from ray.rllib.models import ModelCatalog
from ray.rllib.models.torch.recurrent_net import RecurrentNetwork
from ray.rllib.utils.typing import ModelConfigDict
from ray.tune.registry import register_env

import torch
import torch.nn as nn
import numpy as np
import json
import time
from pathlib import Path
from IPython.display import clear_output

from horse_racing.env.observation import OBS_SIZE
from horse_racing.training.curriculum import get_stage, STAGES

In [ ]:
# --- Custom GRU model for RLlib ---

class GRUJockeyModel(RecurrentNetwork, nn.Module):
    """GRU-based policy for effort+lane action space.

    Architecture:
        obs (63) -> Linear(128) -> ReLU -> GRU(128) -> Linear(64) -> ReLU -> action (2)
    """

    def __init__(self, obs_space, action_space, num_outputs, model_config: ModelConfigDict, name: str, **kwargs):
        nn.Module.__init__(self)
        RecurrentNetwork.__init__(self, obs_space, action_space, num_outputs, model_config, name)

        custom = model_config.get('custom_model_config', {})
        self.obs_dim = obs_space.shape[0]
        self.hidden_size = custom.get('gru_hidden_size', 128)
        self.feature_dim = custom.get('feature_dim', 128)
        self.post_gru_dim = custom.get('post_gru_dim', 64)

        self.feature_extractor = nn.Sequential(
            nn.Linear(self.obs_dim, self.feature_dim),
            nn.ReLU(),
        )
        self.gru = nn.GRU(self.feature_dim, self.hidden_size, batch_first=True)
        self.policy_head = nn.Sequential(
            nn.Linear(self.hidden_size, self.post_gru_dim),
            nn.ReLU(),
            nn.Linear(self.post_gru_dim, num_outputs),
        )
        self.value_head = nn.Sequential(
            nn.Linear(self.hidden_size, self.post_gru_dim),
            nn.ReLU(),
            nn.Linear(self.post_gru_dim, 1),
        )
        self._cur_value = None

    def get_initial_state(self):
        return [torch.zeros(self.hidden_size)]

    def forward_rnn(self, inputs, state, seq_lens):
        features = self.feature_extractor(inputs)
        hidden = state[0].unsqueeze(0)
        gru_out, new_hidden = self.gru(features, hidden)
        new_hidden = new_hidden.squeeze(0)
        policy_out = self.policy_head(gru_out)
        self._cur_value = self.value_head(gru_out).squeeze(-1)
        return policy_out, [new_hidden]

    def value_function(self):
        return self._cur_value.reshape(-1)


ModelCatalog.register_custom_model('gru_jockey', GRUJockeyModel)
print('GRU model registered')

In [ ]:
# --- Environment factory ---

import random as _rng_mod

def make_env(config):
    """Create env based on curriculum stage config."""
    stage_num = config.get('stage_num', 1)
    stage = get_stage(stage_num)
    base_path = config.get('track_base_path', '/content/hr-simulation')

    # Resolve track paths relative to base
    tracks = [f'{base_path}/{t}' for t in stage.tracks]

    if stage.self_play:
        from horse_racing.training.self_play import SelfPlayEnv
        opponent_paths = config.get('opponent_paths', [])
        return SelfPlayEnv(
            tracks=tracks,
            max_steps=stage.max_steps,
            opponent_onnx_paths=opponent_paths,
            min_opponents=stage.min_horses - 1,
            max_opponents=stage.max_horses - 1,
            randomize_jockey_style=stage.randomize_jockey_style,
        )
    else:
        from horse_racing.env.racing_env import HorseRacingEnv
        track = _rng_mod.choice(tracks)
        num_horses = _rng_mod.randint(stage.min_horses, stage.max_horses)
        return HorseRacingEnv(
            track_path=track,
            num_horses=num_horses,
            max_steps=stage.max_steps,
            randomize_horses=stage.randomize_horses,
            randomize_jockey_style=stage.randomize_jockey_style,
        )

register_env('horse_racing_v2', make_env)
print('Environment registered')

In [ ]:
# --- Training runner ---

def train_stage(
    stage_num: int,
    checkpoint_path: str | None = None,
    opponent_paths: list[str] | None = None,
    checkpoint_dir: str = CHECKPOINT_DIR,
    log_every: int = 5,
    save_every: int = 20,
):
    """Train a single curriculum stage. Returns path to best checkpoint."""
    stage = get_stage(stage_num)

    print(f"\n{'='*60}")
    print(f'Stage {stage_num}: {stage.name}')
    print(f'  Tracks: {stage.tracks}')
    print(f'  Horses: {stage.min_horses}-{stage.max_horses}')
    print(f'  Timesteps: {stage.timesteps:,}')
    print(f'  Self-play: {stage.self_play}')
    print(f'  Gate: {stage.gate_metric} >= {stage.gate_threshold}')
    print(f"{'='*60}\n")

    # Detect hardware
    num_gpus = 1 if torch.cuda.is_available() else 0
    import psutil
    total_cpus = psutil.cpu_count()
    num_workers = max(1, min(total_cpus - 1, 6))

    env_config = {
        'stage_num': stage_num,
        'track_base_path': '/content/hr-simulation',
    }
    if opponent_paths:
        env_config['opponent_paths'] = opponent_paths

    config = (
        PPOConfig()
        .api_stack(
            enable_rl_module_and_learner=False,
            enable_env_runner_and_connector_v2=False,
        )
        .environment(
            env='horse_racing_v2',
            env_config=env_config,
        )
        .framework('torch')
        .training(
            gamma=0.99,
            lr=3e-4,
            train_batch_size_per_learner=4096,
            minibatch_size=256,
            num_epochs=6,
            entropy_coeff=0.01,
            clip_param=0.2,
            lambda_=0.95,
        )
        .env_runners(
            num_env_runners=num_workers,
            rollout_fragment_length=64,
        )
        .resources(
            num_gpus=num_gpus,
        )
        .reporting(
            min_sample_timesteps_per_iteration=2000,
        )
    )

    config.model = {
        'custom_model': 'gru_jockey',
        'custom_model_config': {
            'gru_hidden_size': 128,
            'feature_dim': 128,
            'post_gru_dim': 64,
        },
        'max_seq_len': 64,
    }

    algo = config.build()

    if checkpoint_path:
        print(f'Restoring from {checkpoint_path}')
        algo.restore(checkpoint_path)

    # Training loop
    output_path = Path(checkpoint_dir)
    output_path.mkdir(parents=True, exist_ok=True)

    total_timesteps = 0
    best_reward = float('-inf')
    best_checkpoint = None
    iterations = 0
    start_time = time.time()
    history = []

    print(f'Training with {num_workers} workers, {num_gpus} GPU(s)...\n')

    try:
        while total_timesteps < stage.timesteps:
            result = algo.train()
            iterations += 1
            total_timesteps = result.get('timesteps_total', 0)

            mean_reward = result.get('episode_reward_mean', 0)
            max_reward = result.get('episode_reward_max', 0)
            min_reward = result.get('episode_reward_min', 0)
            episodes = result.get('episodes_total', 0)
            ep_len = result.get('episode_len_mean', 0)

            history.append({
                'iter': iterations,
                'timesteps': total_timesteps,
                'mean_reward': mean_reward,
                'max_reward': max_reward,
                'episodes': episodes,
                'ep_len': ep_len,
            })

            if iterations % log_every == 0:
                elapsed = time.time() - start_time
                rate = total_timesteps / elapsed if elapsed > 0 else 0
                pct = total_timesteps / stage.timesteps * 100
                print(
                    f'  [{pct:5.1f}%] Iter {iterations:4d} | '
                    f'ts={total_timesteps:>9,} | '
                    f'reward={mean_reward:7.1f} (max={max_reward:7.1f}) | '
                    f'ep_len={ep_len:5.0f} | '
                    f'{rate:,.0f} ts/s'
                )

            # Save best
            if mean_reward > best_reward:
                best_reward = mean_reward
                best_checkpoint = algo.save(
                    str(output_path / f'stage_{stage_num}_best')
                )
                print(f'  *** New best: reward={best_reward:.1f}')

            # Periodic save to Drive
            if iterations % save_every == 0:
                save_path = algo.save(
                    str(output_path / f'stage_{stage_num}_iter_{iterations}')
                )
                print(f'  Checkpoint saved: {save_path}')

    except KeyboardInterrupt:
        print('\nTraining interrupted by user.')

    # Final save
    final_path = algo.save(str(output_path / f'stage_{stage_num}_final'))

    elapsed = time.time() - start_time
    print(f'\n--- Stage {stage_num} Summary ---')
    print(f'  Total timesteps: {total_timesteps:,}')
    print(f'  Iterations: {iterations}')
    print(f'  Best reward: {best_reward:.1f}')
    print(f'  Time: {elapsed/60:.1f} min')
    print(f'  Final checkpoint: {final_path}')
    print(f'  Best checkpoint: {best_checkpoint}')

    # Save training history
    hist_path = output_path / f'stage_{stage_num}_history.json'
    with open(hist_path, 'w') as f:
        json.dump(history, f, indent=2)

    algo.stop()
    return best_checkpoint or final_path, history

In [ ]:
# --- ONNX export helper ---

def export_stage_to_onnx(checkpoint_path: str, stage_num: int, output_dir: str = ONNX_DIR):
    """Export a checkpoint to ONNX for browser inference + self-play opponents."""
    from horse_racing.training.export_onnx import GRUPolicyWrapper, export_onnx
    from horse_racing.env.observation import OBS_SIZE

    config = (
        PPOConfig()
        .api_stack(
            enable_rl_module_and_learner=False,
            enable_env_runner_and_connector_v2=False,
        )
        .environment(env='horse_racing_v2', env_config={'stage_num': stage_num})
        .framework('torch')
    )
    config.model = {
        'custom_model': 'gru_jockey',
        'custom_model_config': {
            'gru_hidden_size': 128,
            'feature_dim': 128,
            'post_gru_dim': 64,
        },
        'max_seq_len': 64,
    }

    algo = config.build()
    algo.restore(checkpoint_path)

    # Extract weights from RLlib policy
    policy = algo.get_policy()
    rllib_model = policy.model

    # Build ONNX wrapper and copy weights
    wrapper = GRUPolicyWrapper(
        obs_dim=OBS_SIZE,
        hidden_size=128,
        action_dim=2,
        feature_dim=128,
        post_gru_dim=64,
    )

    # Map RLlib model weights -> ONNX wrapper
    wrapper.feature_extractor.weight.data = rllib_model.feature_extractor[0].weight.data.clone()
    wrapper.feature_extractor.bias.data = rllib_model.feature_extractor[0].bias.data.clone()
    wrapper.gru.load_state_dict(rllib_model.gru.state_dict())
    wrapper.post_gru.weight.data = rllib_model.policy_head[0].weight.data.clone()
    wrapper.post_gru.bias.data = rllib_model.policy_head[0].bias.data.clone()
    wrapper.action_head.weight.data = rllib_model.policy_head[2].weight.data.clone()
    wrapper.action_head.bias.data = rllib_model.policy_head[2].bias.data.clone()

    output_path = f'{output_dir}/v2_stage_{stage_num}.onnx'
    export_onnx(wrapper, output_path)

    algo.stop()
    print(f'\nONNX exported: {output_path}')
    return output_path

In [ ]:
# --- Evaluation helper ---

def evaluate_stage(checkpoint_path: str, stage_num: int, num_episodes: int = 50):
    """Run evaluation episodes and compute gate metrics."""
    stage = get_stage(stage_num)

    config = (
        PPOConfig()
        .api_stack(
            enable_rl_module_and_learner=False,
            enable_env_runner_and_connector_v2=False,
        )
        .environment(
            env='horse_racing_v2',
            env_config={
                'stage_num': stage_num,
                'track_base_path': '/content/hr-simulation',
            },
        )
        .framework('torch')
        .env_runners(num_env_runners=0)
        .resources(num_gpus=0)
    )
    config.model = {
        'custom_model': 'gru_jockey',
        'custom_model_config': {
            'gru_hidden_size': 128,
            'feature_dim': 128,
            'post_gru_dim': 64,
        },
        'max_seq_len': 64,
    }

    algo = config.build()
    algo.restore(checkpoint_path)

    env = make_env({
        'stage_num': stage_num,
        'track_base_path': '/content/hr-simulation',
    })

    finishes = 0
    top2 = 0
    wins = 0
    total_reward = 0
    total_progress = 0

    for ep in range(num_episodes):
        obs, info = env.reset(seed=ep)
        state = algo.get_policy().get_initial_state()
        done = False
        ep_reward = 0

        while not done:
            action = algo.compute_single_action(obs, state=state)
            if isinstance(action, tuple):
                action, state, _ = action
            obs, reward, terminated, truncated, info = env.step(action)
            ep_reward += reward
            done = terminated or truncated

        total_reward += ep_reward
        total_progress += info.get('progress', 0)
        if info.get('finished', False):
            finishes += 1
            placement = info.get('placement', 99)
            if placement == 0:
                wins += 1
            if placement <= 1:
                top2 += 1

    algo.stop()

    metrics = {
        'finish_rate': finishes / num_episodes,
        'win_rate': wins / num_episodes,
        'top2_rate': top2 / num_episodes,
        'mean_reward': total_reward / num_episodes,
        'mean_progress': total_progress / num_episodes,
    }

    print(f'\n--- Stage {stage_num} Evaluation ({num_episodes} episodes) ---')
    for k, v in metrics.items():
        gate_marker = ''
        if k == stage.gate_metric:
            passed = v >= stage.gate_threshold
            gate_marker = f'  <-- GATE {"PASSED" if passed else "FAILED"} (need >= {stage.gate_threshold})'
        print(f'  {k}: {v:.3f}{gate_marker}')

    gate_val = metrics.get(stage.gate_metric, 0)
    return gate_val >= stage.gate_threshold, metrics

In [ ]:
# --- Training progress plot ---

import matplotlib.pyplot as plt

def plot_training(history, stage_num):
    """Plot training curves."""
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))

    iters = [h['iter'] for h in history]
    rewards = [h['mean_reward'] for h in history]
    max_rewards = [h['max_reward'] for h in history]
    ep_lens = [h['ep_len'] for h in history]
    timesteps = [h['timesteps'] for h in history]

    axes[0].plot(iters, rewards, label='Mean', alpha=0.7)
    axes[0].plot(iters, max_rewards, label='Max', alpha=0.4)
    axes[0].set_xlabel('Iteration')
    axes[0].set_ylabel('Reward')
    axes[0].set_title(f'Stage {stage_num} — Reward')
    axes[0].legend()
    axes[0].grid(True, alpha=0.3)

    axes[1].plot(iters, ep_lens, alpha=0.7, color='orange')
    axes[1].set_xlabel('Iteration')
    axes[1].set_ylabel('Episode Length')
    axes[1].set_title(f'Stage {stage_num} — Episode Length')
    axes[1].grid(True, alpha=0.3)

    axes[2].plot(timesteps, rewards, alpha=0.7, color='green')
    axes[2].set_xlabel('Timesteps')
    axes[2].set_ylabel('Mean Reward')
    axes[2].set_title(f'Stage {stage_num} — Reward vs Timesteps')
    axes[2].grid(True, alpha=0.3)

    plt.tight_layout()
    plt.savefig(f'{CHECKPOINT_DIR}/stage_{stage_num}_curves.png', dpi=100)
    plt.show()

## 3. Initialize Ray & Resume

In [ ]:
import psutil
import glob

# Pro+ resources: maximize utilization
num_cpus = psutil.cpu_count()
num_gpus = 1 if torch.cuda.is_available() else 0
ram_gb = psutil.virtual_memory().total / 1e9
object_store_gb = min(ram_gb * 0.3, 20.0)

if ray.is_initialized():
    ray.shutdown()

ray.init(
    num_cpus=num_cpus,
    num_gpus=num_gpus,
    object_store_memory=int(object_store_gb * 1e9),
    ignore_reinit_error=True,
)
print(f'Ray initialized: {num_cpus} CPUs, {num_gpus} GPU(s), {object_store_gb:.0f} GB object store')

# --- Scan Drive for existing checkpoints and ONNX models ---
print(f'\n{"="*60}')
print('Existing checkpoints on Drive:')
print(f'{"="*60}')
for stage_num in range(1, 6):
    best = glob.glob(f'{CHECKPOINT_DIR}/stage_{stage_num}_best*')
    final = glob.glob(f'{CHECKPOINT_DIR}/stage_{stage_num}_final*')
    iters = sorted(glob.glob(f'{CHECKPOINT_DIR}/stage_{stage_num}_iter_*'))
    onnx = glob.glob(f'{ONNX_DIR}/v2_stage_{stage_num}.onnx')
    if best or final or iters:
        latest = (final or best or iters)[-1]
        print(f'  Stage {stage_num}: best={len(best)>0}  final={len(final)>0}  iter_checkpoints={len(iters)}  onnx={len(onnx)>0}')
        print(f'           latest: {latest}')
    else:
        print(f'  Stage {stage_num}: (no checkpoints)')

print(f'\n--- Set START_STAGE and checkpoint below, then run the training cell ---')

## 4. Configure Training

Set `START_STAGE` to pick up where you left off after a disconnect.
Set `RESUME_CHECKPOINT` to continue from a specific checkpoint, or leave as `'auto'` to auto-detect the latest one on Drive.

In [ ]:
#@title Training Configuration { run: "auto" }

#@markdown ### Which stage to start from (1-5)
START_STAGE = 1  #@param {type:"integer"}

#@markdown ### Resume checkpoint (or 'auto' to find latest on Drive)
RESUME_CHECKPOINT = 'auto'  #@param {type:"string"}

#@markdown ### Train through all remaining stages?
TRAIN_ALL_REMAINING = True  #@param {type:"boolean"}

# --- Auto-detect checkpoint ---
def find_latest_checkpoint(stage_num):
    """Find the latest checkpoint for a given stage on Drive."""
    candidates = []
    for pattern in [f'stage_{stage_num}_best', f'stage_{stage_num}_final', f'stage_{stage_num}_iter_*']:
        candidates.extend(glob.glob(f'{CHECKPOINT_DIR}/{pattern}'))
    if not candidates:
        return None
    # Prefer best > final > iter
    for tag in ['best', 'final']:
        matches = [c for c in candidates if tag in c]
        if matches:
            return sorted(matches)[-1]
    return sorted(candidates)[-1]

def find_onnx_opponents():
    """Collect all ONNX models exported so far."""
    paths = sorted(glob.glob(f'{ONNX_DIR}/v2_stage_*.onnx'))
    return [p for p in paths if Path(p).exists()]

if RESUME_CHECKPOINT == 'auto':
    # Look for checkpoint from the current stage first (to resume mid-stage),
    # then fall back to previous stage's best (to warm-start a new stage)
    resolved = find_latest_checkpoint(START_STAGE)
    if resolved is None and START_STAGE > 1:
        resolved = find_latest_checkpoint(START_STAGE - 1)
    RESUME_CHECKPOINT = resolved

print(f'Start stage:  {START_STAGE}')
print(f'Checkpoint:   {RESUME_CHECKPOINT or "(fresh start)"}')
print(f'Train all:    {TRAIN_ALL_REMAINING}')
print(f'Opponents:    {find_onnx_opponents() or "(none yet)"}')

## 5. Train

In [ ]:
# --- Main training loop ---
# Runs from START_STAGE through stage 5 (or just START_STAGE if TRAIN_ALL_REMAINING=False)

end_stage = 5 if TRAIN_ALL_REMAINING else START_STAGE
checkpoint = RESUME_CHECKPOINT
all_histories = {}

for stage_num in range(START_STAGE, end_stage + 1):
    stage = get_stage(stage_num)

    # Gather opponent ONNX models for self-play stages
    opponents = find_onnx_opponents() if stage.self_play else None

    # Train
    checkpoint, history = train_stage(
        stage_num=stage_num,
        checkpoint_path=checkpoint,
        opponent_paths=opponents,
    )
    all_histories[stage_num] = history

    # Plot
    plot_training(history, stage_num)

    # Evaluate
    passed, metrics = evaluate_stage(checkpoint, stage_num=stage_num, num_episodes=100)
    print(f'\nStage {stage_num} gate: {"PASSED" if passed else "FAILED"}')

    # Export ONNX (needed as opponent for later stages + browser)
    onnx_path = export_stage_to_onnx(checkpoint, stage_num=stage_num)
    print(f'Exported: {onnx_path}')

    if not passed:
        print(f'\n*** Stage {stage_num} gate FAILED — stopping. Re-run with START_STAGE={stage_num} to retry. ***')
        break

    print(f'\n{"="*60}')
    print(f'Stage {stage_num} complete — advancing to stage {stage_num + 1}')
    print(f'{"="*60}\n')

print('\n--- Training finished ---')
print(f'Last checkpoint: {checkpoint}')

## 6. Download Models

In [ ]:
# Download the best ONNX model to your local machine
from google.colab import files
import glob

# Find the latest exported ONNX
onnx_files = sorted(glob.glob(f'{ONNX_DIR}/*.onnx'))
print('Available ONNX models:')
for f in onnx_files:
    size_kb = Path(f).stat().st_size / 1024
    print(f'  {f} ({size_kb:.1f} KB)')

if onnx_files:
    latest = onnx_files[-1]
    print(f'\nDownloading: {latest}')
    files.download(latest)

## 7. Cleanup

In [ ]:
ray.shutdown()
print('Ray shut down. Training complete.')